In [ ]:
!pip install distrax
!pip install "gymnasium[box2d]"

In [ ]:
import torch
import jax
from flax import linen as nn
import gymnasium as gym
from gymnasium.vector import SyncVectorEnv
import jax.numpy as jnp
from flax.training.train_state import TrainState
import numpy as np

from gymnasium.wrappers import RecordVideo
import wandb
import optax
from flax.core import apply
import distrax

In [ ]:
ENV_NAME = "LunarLander-v3"
ACTORLEARNING_RATES = 3e-4
CRITICLEARNING_RATES = 5e-4
REWARD_DECAY = 0.9998
EVAL_LOOP = 3
TOTAL_ENVS = 10
MAX_ROLLOUT_STEPS = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
print(jax.devices())

In [ ]:
def create_env():
  def _init():
    env = gym.make(id = ENV_NAME)
    return env
  return _init

vec_envs = SyncVectorEnv([create_env() for _ in range(TOTAL_ENVS)])

In [ ]:
vec_envs.single_action_space.n

In [ ]:
state, _ = vec_envs.reset()

In [ ]:
class ActorNetworkClass(nn.Module):

    action_space: int

    @nn.compact
    def __call__(self, x, ):

      x = nn.Dense(features = 256)(x)
      x = nn.relu(x)
      x = nn.Dense(features = 256)(x)
      x = nn.relu(x)
      x = nn.Dense(features = 256)(x)
      x = nn.relu(x)
      x = nn.Dense(features = self.action_space)(x)
      return x

# ==========================================================================================================================================================

class CriticNetworkClass(nn.Module):

    @nn.compact
    def __call__(self, x, ):

      x = nn.Dense(features = 256)(x)
      x = nn.relu(x)
      x = nn.Dense(features = 256)(x)
      x = nn.relu(x)
      x = nn.Dense(features = 1)(x)
      return x

In [ ]:
actornetwork = ActorNetworkClass(action_space=vec_envs.single_action_space.n)  # type: ignore
criticnetwork = CriticNetworkClass()  # type: ignore

In [ ]:
key = jax.random.PRNGKey(0)

In [ ]:
x_dummy = jnp.ones((1, state.shape[-1]))

In [ ]:
actor_variables = actornetwork.init(key, x_dummy)
critic_variables = criticnetwork.init(key, x_dummy)

In [ ]:
actor_optimizers = optax.chain(
    optax.clip_by_global_norm(0.8),
    optax.adam(learning_rate = ACTORLEARNING_RATES)
    )
critic_optimizers = optax.chain(
    optax.clip_by_global_norm(0.8),
    optax.adam(learning_rate = CRITICLEARNING_RATES)
    )

actor_optm_state = actor_optimizers.init(actor_variables['params'])
critic_optm_state = critic_optimizers.init(critic_variables['params'])

In [ ]:
actor_state = TrainState(
    apply_fn = actornetwork.apply,
    params = actor_variables["params"],
    tx = actor_optimizers,
    opt_state = actor_optm_state,
    step=0
)

critic_state = TrainState(
    apply_fn = criticnetwork.apply,
    params = critic_variables["params"],
    tx = critic_optimizers,
    opt_state = critic_optm_state,
    step=0
)

# **training loop**

In [ ]:
state, _ = vec_envs.reset(seed = 42)

In [ ]:
def get_critic_loss(params, all_states, targets):
  x = critic_state.apply_fn(variables={'params': params}, x=all_states).squeeze(-1)
  loss = optax.squared_error(predictions=x, targets=targets)
  return loss.mean()

critic_loss_fn = jax.jit(get_critic_loss)

In [ ]:
def get_actor_loss(params, stored_states, stored_actions, stored_advantages):

    logits = actor_state.apply_fn(variables={'params': params}, x = stored_states)
    log_probs_all = jax.nn.log_softmax(logits)
    # Get log probabilities for the actions that were actually taken
    log_probs_taken_actions = jnp.take_along_axis(log_probs_all, jnp.expand_dims(stored_actions, axis=-1), axis=-1).squeeze(-1)
    loss_value = - jnp.mean(log_probs_taken_actions*stored_advantages)
    return loss_value

actor_loss_fn = jax.jit(get_actor_loss)

In [ ]:
def wandb_runs():
    wandb.login(key=" ")
    run = wandb.init(
      project="Jax-Project",
      name = f"{ENV_NAME}"
      )
    return run

runs = wandb_runs()

In [ ]:
import time
def evaluation(actor_state, rec_video = False, seed=None):

  eval_env = gym.make(ENV_NAME, render_mode = 'rgb_array')

  if rec_video:
    video_dir = f"videos/{int(time.time())}"
    eval_env = RecordVideo(eval_env, video_folder=video_dir, episode_trigger=lambda ep: True)

  total_reward = 0
  total_step = 0

  for _ in range(EVAL_LOOP):

    state = eval_env.reset()[0]
    done = False

    while not done:
      logits = actor_state.apply_fn(variables={'params': actor_state.params}, x=state)
      action = logits.argmax().item()
      next_state, reward, terminated, truncation, _ = eval_env.step(action)
      done = terminated or truncation
      state = next_state
      total_reward += float(reward)
      total_step += 1

  average_reward = total_reward / EVAL_LOOP
  average_step = total_step / EVAL_LOOP
  eval_env.close()

  return average_reward, average_step

In [ ]:
# evaluation(actor_state, True)

In [ ]:
TOTAL_STEPS = 1_000_000
EVAL_STEPS = 10_000
global_steps = 0

state, _ = vec_envs.reset(seed=42)
training_rewards = jnp.zeros((TOTAL_ENVS,))
key = jax.random.PRNGKey(0)

while global_steps <= TOTAL_STEPS:

    running_steps = 0
    all_states, all_next_states = [], []
    all_rewards, all_dones, all_actions = [], [], []

    while running_steps < MAX_ROLLOUT_STEPS:

        all_logits = actor_state.apply_fn(
            variables={"params": actor_state.params},
            x=state
        )
        dist = distrax.Categorical(logits=all_logits)

        key, subkey = jax.random.split(key)
        sampled_action = dist.sample(seed=subkey)

        next_state, reward, terminated, truncated, info = vec_envs.step(
            np.array(sampled_action)
        )

        done = terminated | truncated

        training_rewards += reward
        for i, check_done in enumerate(terminated):
            if check_done:
                runs.log({"training-reward": training_rewards[i]}, step=global_steps)
                training_rewards = training_rewards.at[i].set(0)

        all_states.append(state)
        all_next_states.append(next_state)
        all_rewards.append(reward)
        all_dones.append(terminated)
        all_actions.append(sampled_action)

        state = next_state

        if global_steps % EVAL_STEPS == 0 and global_steps > 0:
            average_reward, average_step = evaluation(actor_state, False)
            runs.log(
                {
                    "average-reward": average_reward,
                    "average-step": average_step,
                },
                step=global_steps
            )

        running_steps += 1
        global_steps += TOTAL_ENVS

    stacked_all_states = jnp.stack(all_states)          # [T, N, ...]
    stacked_all_nxt_st = jnp.stack(all_next_states)     # [T, N, ...]
    stacked_all_actions = jnp.stack(all_actions)        # [T, N]
    stacked_all_rewards = jnp.stack(all_rewards)        # [T, N]
    stacked_all_dones = jnp.stack(all_dones)            # [T, N]

    values_current = critic_state.apply_fn(
        variables={"params": critic_state.params},
        x=stacked_all_states
    ).squeeze(-1)                                       # [T, N]

    values_next_st = critic_state.apply_fn(
        variables={"params": critic_state.params},
        x=stacked_all_nxt_st
    ).squeeze(-1)                                       # [T, N]

    # Bootstrapped returns
    stacked_all_targets = jnp.zeros_like(stacked_all_rewards)
    for env_idx in range(TOTAL_ENVS):
        next_return = values_next_st[-1, env_idx]
        if stacked_all_dones[-1, env_idx]:
            next_return = 0.0

        for t in reversed(range(MAX_ROLLOUT_STEPS)):
            current_return = (
                stacked_all_rewards[t, env_idx]
                + REWARD_DECAY * (1.0 - stacked_all_dones[t, env_idx]) * next_return
            )
            next_return = current_return
            stacked_all_targets = stacked_all_targets.at[t, env_idx].set(current_return)

    stacked_all_targets = jax.lax.stop_gradient(stacked_all_targets)

    # Use the same target for actor + critic
    advantage = stacked_all_targets - values_current
    advantage = (advantage - jnp.mean(advantage)) / (jnp.std(advantage) + 1e-8)
    advantage = jax.lax.stop_gradient(advantage)

    # ============================ critic update ============================
    critic_loss_value, critic_grad = jax.value_and_grad(critic_loss_fn)(
        critic_state.params,
        jnp.array(stacked_all_states),
        stacked_all_targets
    )
    critic_state = critic_state.apply_gradients(grads=critic_grad)

    # ============================ actor update =============================
    actor_loss_fn_value, actor_grad = jax.value_and_grad(actor_loss_fn)(
        actor_state.params,
        jnp.array(stacked_all_states),
        jnp.array(stacked_all_actions),
        advantage
    )
    actor_state = actor_state.apply_gradients(grads=actor_grad)

    runs.log(
        {
            "actor-loss": actor_loss_fn_value,
            "critic-loss": critic_loss_value,
            "advantages": jnp.mean(advantage),
            "global-steps": global_steps,
        },
        step=global_steps
    )